# Day 7: Week 1 Capstone
## The Distribution Profile of Hong Kong Markets

This notebook summarizes Week 1 of the systematic macro quant curriculum by analyzing the Hang Seng Index (`^HSI`) through the lens of distributions, volatility, tail risk, skewness, and correlation.

The core research question is:

> What does the distribution of HSI returns tell a systematic macro investor about Hong Kong equity-market risk?

The analysis uses real market data from Yahoo Finance and focuses on five questions:

1. Does HSI historically reward investors on average?
2. Is HSI a calm or volatile equity market?
3. Can a normal distribution safely describe HSI risk?
4. Are HSI extreme moves more dangerous on the downside?
5. Does HSI diversify a global macro portfolio?

## Setup

Run the next cell in Google Colab if `yfinance` is not already installed. The notebook uses only standard research libraries: `pandas`, `numpy`, `scipy`, `matplotlib`, `seaborn`, and `yfinance`.

In [ ]:
# Uncomment this line in Google Colab if yfinance is not installed.
# !pip install yfinance -q

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, kurtosis, skew
from IPython.display import display, Markdown
import matplotlib.ticker as mtick

warnings.filterwarnings("ignore")

PERIOD = "10y"
TRADING_DAYS = 252
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")

TICKERS = {
    "HSI": "^HSI",
    "S&P 500": "^GSPC",
    "Nasdaq": "^IXIC",
    "Gold": "GLD",
    "Oil": "CL=F",
    "US 10Y Yield": "^TNX",
}


def pct(x):
    return f"{x:.2%}"


def pct4(x):
    return f"{x:.4%}"


def number(x):
    return f"{x:,.2f}"


def clean_close(downloaded, tickers):
    close = downloaded["Close"]
    reverse_names = {symbol: name for name, symbol in tickers.items()}
    if isinstance(close, pd.Series):
        return close.to_frame(name=list(tickers.keys())[0])
    return close.rename(columns=reverse_names).dropna(how="all")

## Data

The capstone uses 10 years of daily data to capture multiple regimes instead of relying only on the most recent market environment.

In [ ]:
raw_data = yf.download(
    list(TICKERS.values()),
    period=PERIOD,
    auto_adjust=True,
    progress=False,
)

prices = clean_close(raw_data, TICKERS)
returns = prices.pct_change().dropna(how="all")
hsi_prices = prices["HSI"].dropna()
hsi_returns = returns["HSI"].dropna()

data_summary = pd.DataFrame({
    "Item": ["Sample start", "Sample end", "HSI observations", "Assets used"],
    "Value": [
        hsi_returns.index.min().strftime("%Y-%m-%d"),
        hsi_returns.index.max().strftime("%Y-%m-%d"),
        f"{len(hsi_returns):,}",
        ", ".join(returns.columns.tolist()),
    ],
})

display(data_summary)

## Q1. Does HSI Historically Reward Investors On Average?

This section measures the basic return profile: mean return, median return, probability of positive days, annualized return, CAGR, and the best/worst observed daily moves.

In [ ]:
mean_daily = hsi_returns.mean()
median_daily = hsi_returns.median()
daily_vol = hsi_returns.std()
annualized_return = (1 + mean_daily) ** TRADING_DAYS - 1
annualized_volatility = daily_vol * np.sqrt(TRADING_DAYS)
positive_day_probability = (hsi_returns > 0).mean()
worst_day = hsi_returns.min()
best_day = hsi_returns.max()
prob_loss_more_than_1 = (hsi_returns < -0.01).mean()
prob_between_minus1_plus1 = ((hsi_returns > -0.01) & (hsi_returns < 0.01)).mean()
prob_gain_more_than_2 = (hsi_returns > 0.02).mean()
years = (hsi_prices.index[-1] - hsi_prices.index[0]).days / 365.25
cagr = (hsi_prices.iloc[-1] / hsi_prices.iloc[0]) ** (1 / years) - 1

q1_table = pd.DataFrame({
    "Metric": [
        "Mean daily return",
        "Median daily return",
        "Daily volatility",
        "Mean-based annualized return",
        "CAGR",
        "Annualized volatility",
        "Positive day probability",
        "Worst day",
        "Best day",
        "P(return < -1%)",
        "P(-1% < return < +1%)",
        "P(return > +2%)",
    ],
    "Value": [
        pct4(mean_daily),
        pct4(median_daily),
        pct(daily_vol),
        pct(annualized_return),
        pct(cagr),
        pct(annualized_volatility),
        pct(positive_day_probability),
        pct(worst_day),
        pct(best_day),
        pct(prob_loss_more_than_1),
        pct(prob_between_minus1_plus1),
        pct(prob_gain_more_than_2),
    ],
})

display(q1_table)

**Answer:** HSI has historically delivered a positive but weak reward over this sample. The mean and median daily returns were positive, the positive-day probability was above 51%, and CAGR was positive. However, the reward was small relative to the risk: annualized volatility was high while realized CAGR was modest. The market also had meaningful downside risk, with the worst daily loss larger in magnitude than the best daily gain and a material share of days losing more than 1%. Overall, HSI offered positive but weak risk-adjusted compensation rather than a clean, attractive buy-and-hold profile.

## Q2. Is HSI A Calm Or Volatile Equity Market?

This section compares HSI volatility with the S&P 500 and Nasdaq, then shows how HSI volatility changes through time.

In [ ]:
equity_assets = ["HSI", "S&P 500", "Nasdaq"]
equity_returns = returns[equity_assets].dropna()

daily_volatility = equity_returns.std()
annualized_volatility = daily_volatility * np.sqrt(TRADING_DAYS)

vol_table = pd.DataFrame({
    "Daily Volatility": daily_volatility,
    "Annualized Volatility": annualized_volatility,
}).sort_values("Annualized Volatility", ascending=False)

vol_display = vol_table.copy()
vol_display["Daily Volatility"] = vol_display["Daily Volatility"].map(pct)
vol_display["Annualized Volatility"] = vol_display["Annualized Volatility"].map(pct)

display(vol_display)

hsi_vs_spx = annualized_volatility["HSI"] / annualized_volatility["S&P 500"] - 1
print(f"HSI is {hsi_vs_spx:.2%} more volatile than the S&P 500 on an annualized basis.")

In [ ]:
rolling_hsi_vol = hsi_returns.rolling(30).std() * np.sqrt(TRADING_DAYS)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(rolling_hsi_vol, color="#1f77b4", linewidth=1.8, label="HSI 30-day rolling annualized volatility")
ax.axvspan(pd.Timestamp("2020-02-01"), pd.Timestamp("2020-04-30"), color="red", alpha=0.15, label="COVID crash period")
ax.set_title("HSI Rolling 30-Day Annualized Volatility")
ax.set_xlabel("Date")
ax.set_ylabel("30-day rolling annualized volatility")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
ax.grid(True, alpha=0.35)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "q2_hsi_rolling_volatility.png", dpi=150)
plt.show()

**Answer:** HSI shows moderately high volatility. Its annualized volatility is higher than the S&P 500 and close to Nasdaq, which means HSI carries substantial equity-market risk. The main issue is not volatility alone; it is that HSI's realized return and CAGR are low relative to its volatility. This suggests investors were not strongly compensated for the risk they took. Overall, HSI appears riskier than the S&P 500 on a volatility basis, and its weak return-to-risk profile makes it unattractive as a simple buy-and-hold exposure unless there is a specific macro signal or diversification reason to hold it.

## Q3. Can A Normal Distribution Safely Describe HSI Risk?

This section fits a normal distribution to HSI daily returns, compares model probabilities with historical frequencies, and then checks whether 3-sigma events occur more often than a normal model predicts.

In [ ]:
thresholds = {
    "Lose more than 1%": -0.01,
    "Lose more than 3%": -0.03,
    "Gain more than 5%": 0.05,
}

risk_rows = []
for label, threshold in thresholds.items():
    z_score = (threshold - mean_daily) / daily_vol
    if threshold < mean_daily:
        normal_probability = norm.cdf(z_score)
        historical_probability = (hsi_returns < threshold).mean()
    else:
        normal_probability = 1 - norm.cdf(z_score)
        historical_probability = (hsi_returns > threshold).mean()

    risk_rows.append({
        "Scenario": label,
        "Threshold": threshold,
        "Z-score": z_score,
        "Normal probability": normal_probability,
        "Historical probability": historical_probability,
        "Historical / Normal": historical_probability / normal_probability if normal_probability > 0 else np.nan,
    })

risk_table = pd.DataFrame(risk_rows)
risk_display = risk_table.copy()
risk_display["Threshold"] = risk_display["Threshold"].map(pct)
risk_display["Z-score"] = risk_display["Z-score"].map(lambda x: f"{x:.2f}")
risk_display["Normal probability"] = risk_display["Normal probability"].map(pct)
risk_display["Historical probability"] = risk_display["Historical probability"].map(pct)
risk_display["Historical / Normal"] = risk_display["Historical / Normal"].map(lambda x: f"{x:.1f}x")

display(risk_display)

In [ ]:
x = np.linspace(hsi_returns.min(), hsi_returns.max(), 600)
normal_curve = norm.pdf(x, mean_daily, daily_vol)

fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(hsi_returns, bins=70, density=True, alpha=0.62, edgecolor="white", label="Actual HSI returns")
ax.plot(x, normal_curve, color="red", linewidth=2.2, label="Fitted normal distribution")
ax.axvline(-0.03, color="black", linestyle="--", linewidth=1.5, label="-3% loss threshold")
ax.set_title("HSI Daily Returns vs Fitted Normal Distribution")
ax.set_xlabel("Daily return")
ax.set_ylabel("Density")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
ax.grid(True, alpha=0.35)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "q3_hsi_returns_vs_normal.png", dpi=150)
plt.show()

In [ ]:
raw_kurtosis = kurtosis(hsi_returns, fisher=False)
excess_kurtosis = kurtosis(hsi_returns, fisher=True)
upper_3sigma = mean_daily + 3 * daily_vol
lower_3sigma = mean_daily - 3 * daily_vol
actual_3sigma_days = ((hsi_returns > upper_3sigma) | (hsi_returns < lower_3sigma)).sum()
actual_3sigma_frequency = ((hsi_returns > upper_3sigma) | (hsi_returns < lower_3sigma)).mean()
normal_3sigma_frequency = 2 * (1 - norm.cdf(3))
expected_3sigma_days = normal_3sigma_frequency * len(hsi_returns)
danger_ratio = actual_3sigma_days / expected_3sigma_days

fat_tail_table = pd.DataFrame([{
    "Asset": "HSI",
    "Kurtosis": raw_kurtosis,
    "Excess Kurtosis": excess_kurtosis,
    "Actual 3-sigma days": actual_3sigma_days,
    "Expected 3-sigma days under normal": expected_3sigma_days,
    "Actual 3-sigma frequency": actual_3sigma_frequency,
    "Normal 3-sigma frequency": normal_3sigma_frequency,
    "Danger ratio": danger_ratio,
}])

fat_tail_display = fat_tail_table.copy()
fat_tail_display["Kurtosis"] = fat_tail_display["Kurtosis"].map(lambda x: f"{x:.2f}")
fat_tail_display["Excess Kurtosis"] = fat_tail_display["Excess Kurtosis"].map(lambda x: f"{x:.2f}")
fat_tail_display["Expected 3-sigma days under normal"] = fat_tail_display["Expected 3-sigma days under normal"].map(lambda x: f"{x:.1f}")
fat_tail_display["Actual 3-sigma frequency"] = fat_tail_display["Actual 3-sigma frequency"].map(pct)
fat_tail_display["Normal 3-sigma frequency"] = fat_tail_display["Normal 3-sigma frequency"].map(pct)
fat_tail_display["Danger ratio"] = fat_tail_display["Danger ratio"].map(lambda x: f"{x:.1f}x")

display(fat_tail_display)

**Answer:** No, a normal distribution cannot safely describe HSI risk on its own. It may approximate the center of the daily return distribution, where most ordinary market moves occur, but it underestimates tail risk. The historical frequency of large losses and 3-sigma events is higher than the fitted normal model predicts, indicating fat tails and non-normal return behavior. Therefore, the normal distribution is useful as a baseline, but not sufficient for serious HSI risk analysis.

## Q4. Are HSI Extreme Moves More Dangerous On The Downside?

This section measures skewness and compares upside versus downside extremes.

In [ ]:
skewness = skew(hsi_returns)
left_tail_1 = hsi_returns.quantile(0.01)
right_tail_99 = hsi_returns.quantile(0.99)
left_tail_5 = hsi_returns.quantile(0.05)
right_tail_95 = hsi_returns.quantile(0.95)

skew_table = pd.DataFrame({
    "Metric": [
        "Skewness",
        "Worst day",
        "Best day",
        "1st percentile return",
        "99th percentile return",
        "5th percentile return",
        "95th percentile return",
    ],
    "Value": [
        f"{skewness:.2f}",
        pct(worst_day),
        pct(best_day),
        pct(left_tail_1),
        pct(right_tail_99),
        pct(left_tail_5),
        pct(right_tail_95),
    ],
})

display(skew_table)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(hsi_returns, bins=70, alpha=0.7, edgecolor="white", color="#4c78a8")
ax.axvline(0, color="black", linestyle="--", linewidth=1.2, label="0% return")
ax.axvline(left_tail_5, color="red", linestyle="--", linewidth=1.4, label="5th percentile")
ax.axvline(right_tail_95, color="green", linestyle="--", linewidth=1.4, label="95th percentile")
ax.set_title("HSI Daily Return Distribution: Downside vs Upside Tail")
ax.set_xlabel("Daily return")
ax.set_ylabel("Number of days")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend()
ax.grid(True, alpha=0.35)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "q4_hsi_skewness_tails.png", dpi=150)
plt.show()

**Answer:** HSI shows downside asymmetry, indicated by negative skewness and a worst daily loss that is larger in magnitude than the best daily gain. This means extreme moves are more dangerous on the downside than on the upside, creating meaningful left-tail or crash risk. Given HSI's low CAGR and modest average return, investors were not strongly compensated for this downside asymmetry. Therefore, HSI should be treated as a risky exposure that requires careful position sizing and tail-risk awareness.

## Q5. Does HSI Diversify A Global Macro Portfolio?

This section checks full-sample correlations and then tests whether the HSI/S&P 500 relationship is stable through time.

In [ ]:
macro_returns = returns[["HSI", "S&P 500", "Nasdaq", "Gold", "Oil", "US 10Y Yield"]].dropna()
correlation_matrix = macro_returns.corr()

display(correlation_matrix.round(2))

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Macro Asset Correlation Matrix")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "q5_macro_correlation_matrix.png", dpi=150)
plt.show()

In [ ]:
rolling_corr = macro_returns["HSI"].rolling(60).corr(macro_returns["S&P 500"])

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(rolling_corr, color="#1f77b4", linewidth=1.7, label="60-day rolling correlation: HSI vs S&P 500")
ax.axvspan(pd.Timestamp("2020-02-01"), pd.Timestamp("2020-04-30"), color="red", alpha=0.15, label="COVID crash period")
ax.axhline(0, color="black", linewidth=1)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="0.5 correlation")
ax.set_title("Rolling Correlation: HSI vs S&P 500")
ax.set_xlabel("Date")
ax.set_ylabel("Correlation")
ax.legend()
ax.grid(True, alpha=0.35)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "q5_hsi_sp500_rolling_correlation.png", dpi=150)
plt.show()

rolling_summary = pd.DataFrame({
    "Metric": [
        "Full-sample HSI/S&P 500 correlation",
        "Minimum 60-day rolling correlation",
        "Maximum 60-day rolling correlation",
        "Latest 60-day rolling correlation",
    ],
    "Value": [
        f"{correlation_matrix.loc['HSI', 'S&P 500']:.2f}",
        f"{rolling_corr.min():.2f}",
        f"{rolling_corr.max():.2f}",
        f"{rolling_corr.dropna().iloc[-1]:.2f}",
    ],
})

display(rolling_summary)

**Answer:** HSI can provide some diversification to a global portfolio on average, because its full-sample correlations with assets such as the S&P 500, oil, gold, and US 10Y yields were generally low. However, this diversification benefit is unstable. The rolling correlation with the S&P 500 can rise sharply in certain regimes, showing that HSI can become more synchronized with global equities. Therefore, HSI should not be treated as a reliably diversifying asset; its diversification benefit must be stress-tested alongside its high standalone volatility and downside risk.

## Final Summary

HSI offers exposure to a market with relatively low average correlation to some global macro assets, suggesting some diversification value. However, its standalone risk profile is challenging: volatility is high relative to average return, tail risk is meaningful, downside skew is present, and correlations with global equities can rise during certain regimes. A systematic macro investor should therefore avoid sizing HSI purely based on average returns and should instead account for volatility, fat tails, downside asymmetry, and changing correlation regimes.